# BASELINE 1: LFCC + LCNN (OFFICIAL GITHUB CLONE)
Notebook ini berisi implementasi baseline pertama untuk riset Audio Anti-Spoofing (ASVspoof) menggunakan arsitektur **LFCC (Linear Frequency Cepstral Coefficients)** dan **LCNN (Light CNN)**.

Notebook ini disusun secara modular dan reproducible untuk memenuhi standar publikasi ilmiah. Seluruh tahapan dari prapemrosesan, integrasi model, pelatihan, hingga evaluasi metrik akhir diimplementasikan di sini.

## 1. Determinisme & Pengaturan Random Seed
Agar hasil eksperimen bersifat deterministik dan dapat diulang (*reproducible*), kita perlu mengunci *seed* acak untuk Python, NumPy, dan PyTorch.

In [ ]:
import os
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # Multi-GPU
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(42)
print("Determinisme aktif. Random seed diatur ke 42.")

## 2. Tempat Import Dataset (Dataloader)
> **Catatan Penting:** Gunakan sel di bawah ini untuk memuat dataset audio Anda (misalnya ASVspoof 2019 LA). 
> Anda dapat mengganti fungsi pemuatan berkas di bawah ini sesuai struktur folder dataset lokal/Kaggle Anda.

In [ ]:
# =====================================================================
# TEMPAT UNTUK IMPORT DATASET & INISIALISASI DATALOADER
# =====================================================================
from torch.utils.data import Dataset, DataLoader
import torchaudio

class AudioASVDataset(Dataset):
    """
    Dataset PyTorch kustom untuk Audio Anti-Spoofing.
    Menerima file protokol dan folder wav untuk memuat audio.
    """
    def __init__(self, protocols_file, wav_dir, max_len=64000, transform=None):
        """
        Args:
            protocols_file (str): Path ke file protokol (cth: ASVspoof2019.LA.cm.train.trl.txt)
            wav_dir (str): Path ke direktori berisi berkas wav
            max_len (int): Panjang maksimum sampel audio (4 detik pada 16kHz = 64000 sampel)
            transform (callable, optional): Fungsi ekstraksi fitur (misal: LFCC)
        """
        self.wav_dir = wav_dir
        self.max_len = max_len
        self.transform = transform
        self.file_list = []
        self.labels = []
        
        # Contoh membaca berkas protokol ASVspoof
        if protocols_file and os.path.exists(protocols_file):
            print(f"Membaca protokol dari {protocols_file}...")
            with open(protocols_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        file_name = parts[1]
                        key = parts[4]
                        # 1 untuk bonafide, 0 untuk spoof
                        label = 1 if key == 'bonafide' else 0
                        self.file_list.append(file_name)
                        self.labels.append(label)
            print(f"Berhasil memuat {len(self.file_list)} entri dataset.")
        else:
            print(f"PERINGATAN: Berkas protokol tidak ditemukan di '{protocols_file}'.")
            print("Membuat DUMMY dataset untuk pengujian...")
            # Membuat dummy data agar notebook tetap bisa dijalankan untuk demo/testing
            self.file_list = [f"dummy_sample_{i}" for i in range(100)]
            self.labels = [random.choice([0, 1]) for _ in range(100)]

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        label = self.labels[idx]
        
        wav_path = os.path.join(self.wav_dir, f"{file_name}.wav")
        
        if os.path.exists(wav_path):
            waveform, sr = torchaudio.load(wav_path)
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                waveform = resampler(waveform)
            waveform = waveform.squeeze(0)
        else:
            # Menggunakan derau acak sebagai dummy jika file tidak ditemukan
            waveform = torch.randn(self.max_len)
            
        # Pading atau potong agar ukuran waveform seragam
        if waveform.shape[0] < self.max_len:
            pad_len = self.max_len - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
        else:
            waveform = waveform[:self.max_len]
            
        # Terapkan transformasi (ekstraksi fitur LFCC) jika disediakan
        if self.transform:
            feature = self.transform(waveform) # Output shape: (1, n_lfcc, time_frames)
            return feature, label
            
        # Jika tidak ada transform, kembalikan waveform mentah dengan dimensi channel
        return waveform.unsqueeze(0), label

# SILAKAN EDIT PATH DI BAWAH INI SESUAI DENGAN DATASET LOKAL ATAU KAGGLE ANDA
# Contoh path Kaggle: "/kaggle/input/asvspoof-2019-la/ASVspoof2019_LA_train/protocols/ASVspoof2019.LA.cm.train.trl.txt"
PATH_PROTOKOL_TRAIN = "D:/Lomba/Gemastik 2026/Data Mining/panduan/dataset/ASVspoof2019.LA.cm.train.trl.txt"
PATH_WAV_DIR_TRAIN = "D:/Lomba/Gemastik 2026/Data Mining/panduan/dataset/train/wav"

PATH_PROTOKOL_DEV = "D:/Lomba/Gemastik 2026/Data Mining/panduan/dataset/ASVspoof2019.LA.cm.dev.trl.txt"
PATH_WAV_DIR_DEV = "D:/Lomba/Gemastik 2026/Data Mining/panduan/dataset/dev/wav"

print("Inisialisasi dataset selesai. Siap dipasangkan dengan fitur LFCC.")

## 3. Ekstraksi Fitur LFCC (Front-End)
Di bagian ini, kita mendefinisikan kelas ekstraksi fitur **LFCC** menggunakan pustaka `torchaudio`.
Input berupa sinyal audio berukuran 16kHz dan outputnya adalah representasi spektogram LFCC 2D. Kami juga menambahkan fungsi padding/truncating pada dimensi waktu agar setiap batch memiliki ukuran frame yang sama.

In [ ]:
import torch.nn as nn
import torchaudio.transforms as T

class LFCCFeatureExtractor(nn.Module):
    """
    Ekstraktor fitur Linear Frequency Cepstral Coefficients (LFCC) menggunakan torchaudio.
    """
    def __init__(self, sample_rate=16000, n_filter=40, n_lfcc=20, n_fft=512, hop_length=160, win_length=400):
        super(LFCCFeatureExtractor, self).__init__()
        self.lfcc_transform = T.LFCC(
            sample_rate=sample_rate,
            n_filter=n_filter,
            f_min=0.0,
            f_max=sample_rate / 2.0,
            n_lfcc=n_lfcc,
            speckwargs={
                "n_fft": n_fft,
                "win_length": win_length,
                "hop_length": hop_length,
                "center": True
            }
        )

    def forward(self, x):
        # Input x: (batch_size, num_samples) atau (num_samples,)
        # Output: (batch_size, 1, n_lfcc, time_frames) atau (1, n_lfcc, time_frames)
        lfcc = self.lfcc_transform(x)
        # Menambahkan dimensi channel jika belum ada
        if lfcc.ndim == 2:
            lfcc = lfcc.unsqueeze(0) # (1, n_lfcc, time_frames)
        elif lfcc.ndim == 3:
            lfcc = lfcc.unsqueeze(1) # (batch_size, 1, n_lfcc, time_frames)
        return lfcc

# Inisialisasi ekstraktor fitur
lfcc_extractor = LFCCFeatureExtractor(n_lfcc=20)
print("Fungsi ekstraksi fitur LFCC berhasil dibuat.")

## 4. Pendefinisian Arsitektur LCNN & Wrapper (Back-End)
Di bagian ini, kita mendefinisikan arsitektur **Light CNN (LCNN)** (berdasarkan Wu et al., 2018) secara langsung di dalam notebook agar kode bersifat mandiri (*self-contained*) dan siap dijalankan di Kaggle atau Google Colab tanpa membutuhkan kloning repositori luar.

Karena arsitektur asli dikembangkan untuk gambar wajah berukuran tertentu, kita membuat **wrapper** (`LCNNForAudioAntiSpoofing`) untuk:
1. Melakukan adaptive average pooling sebelum fully connected layer agar model dapat menerima ukuran input LFCC berapapun secara adaptif tanpa terjadinya *dimension mismatch*.
2. Memastikan lapisan keluaran (*classifier head*) menghasilkan 2 kelas (*Bona fide* vs *Spoof*).

In [ ]:
# =====================================================================
# ARSITEKTUR ASLI LIGHT CNN (LCNN) - Wu et al., 2018
# =====================================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class mfm(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, type=1):
        super(mfm, self).__init__()
        self.out_channels = out_channels
        if type == 1:
            self.filter = nn.Conv2d(in_channels, 2*out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        else:
            self.filter = nn.Linear(in_channels, 2*out_channels)

    def forward(self, x):
        x = self.filter(x)
        out = torch.split(x, self.out_channels, 1)
        return torch.max(out[0], out[1])

class group(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(group, self).__init__()
        self.conv_a = mfm(in_channels, in_channels, 1, 1, 0)
        self.conv   = mfm(in_channels, out_channels, kernel_size, stride, padding)

    def forward(self, x):
        x = self.conv_a(x)
        x = self.conv(x)
        return x

class resblock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(resblock, self).__init__()
        self.conv1 = mfm(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.conv2 = mfm(in_channels, out_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        res = x
        out = self.conv1(x)
        out = self.conv2(out)
        out = out + res
        return out

class network_9layers(nn.Module):
    def __init__(self, num_classes=79077):
        super(network_9layers, self).__init__()
        self.features = nn.Sequential(
            mfm(1, 48, 5, 1, 2), 
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True), 
            group(48, 96, 3, 1, 1), 
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True),
            group(96, 192, 3, 1, 1),
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True), 
            group(192, 128, 3, 1, 1),
            group(128, 128, 3, 1, 1),
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True),
            )
        self.fc1 = mfm(8*8*128, 256, type=0)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = F.dropout(x, training=self.training)
        out = self.fc2(x)
        return out, x

def LightCNN_9Layers(**kwargs):
    model = network_9layers(**kwargs)
    return model

# =====================================================================
# WRAPPER LCNN AGAR ADAPTIF TERHADAP INPUT LFCC AUDIO
# =====================================================================
class LCNNForAudioAntiSpoofing(nn.Module):
    """
    Wrapper untuk LightCNN agar adaptif terhadap input spectrogram LFCC
    dan mengklasifikasikan ke dalam 2 kelas (Bona fide vs Spoof).
    """
    def __init__(self, num_classes=2):
        super(LCNNForAudioAntiSpoofing, self).__init__()
        # Inisialisasi model LCNN 9-Layers asli
        self.lcnn = LightCNN_9Layers(num_classes=num_classes)
        
        # Adaptive pooling untuk memastikan output conv block berukuran (8, 8)
        # sebelum diumpankan ke FC1, terlepas dari berapa dimensi LFCC dan time frames.
        self.adaptive_pool = nn.AdaptiveAvgPool2d((8, 8))
        
        # Mengubah classifier head terakhir untuk menghasilkan 2 kelas
        self.lcnn.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Input x shape: (batch_size, 1, n_lfcc, time_frames)
        
        # Ekstraksi fitur CNN
        x = self.lcnn.features(x)    # Output shape: (batch_size, 128, H_out, W_out)
        
        # Reduksi dimensi dengan adaptive pool ke (8, 8) sesuai ukuran fc1 asli
        x = self.adaptive_pool(x)    # Output shape: (batch_size, 128, 8, 8)
        
        # Flattening
        x = x.view(x.size(0), -1)    # Output shape: (batch_size, 128 * 8 * 8) = (batch_size, 8192)
        
        # FC1 layer
        x = self.lcnn.fc1(x)         # Output shape: (batch_size, 256)
        
        # Dropout
        x = F.dropout(x, training=self.training)
        
        # Classifier logits
        logits = self.lcnn.fc2(x)    # Output shape: (batch_size, 2)
        
        return logits

# Inisialisasi model baseline
model = LCNNForAudioAntiSpoofing(num_classes=2)
print("Model LCNN dan Wrapper berhasil didefinisikan secara mandiri.")

## 5. Metrik Evaluasi: Equal Error Rate (EER) & tandem Detection Cost Function (t-DCF)
Bagian ini berisi fungsi untuk menghitung metrik evaluasi yang disyaratkan:
1. **EER:** Dihitung menggunakan `sklearn.metrics.roc_curve` dan root-finding `scipy.optimize.brentq`.
2. **min t-DCF:** Kami menyediakan fungsi perhitungan min t-DCF standar ASVspoof. Karena evaluasi t-DCF tandem membutuhkan skor dari sistem ASV, kami menyertakan parameter standar ASVspoof 2019 LA sebagai default agar evaluasi tetap dapat berjalan meskipun tanpa file skor ASV eksternal.

In [ ]:
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def compute_eer_scipy(bonafide_scores, spoof_scores):
    """
    Menghitung Equal Error Rate (EER) menggunakan scipy.optimize.brentq dan sklearn.metrics.roc_curve.
    
    Args:
        bonafide_scores (list/np.ndarray): Skor untuk kelas bona fide (semakin tinggi semakin bona fide)
        spoof_scores (list/np.ndarray): Skor untuk kelas spoofing (semakin rendah semakin spoof)
    Returns:
        float: Nilai EER (dalam desimal, kalikan 100 untuk persentase)
    """
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    y_score = np.concatenate([bonafide_scores, spoof_scores])
    
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    
    # EER dicapai saat FPR == FNR (atau FPR + TPR - 1 == 0)
    eer = brentq(lambda x: interp1d(fpr, tpr)(x) + x - 1., 0., 1.)
    return eer

def compute_asvspoof19_min_tdcf(bonafide_scores, spoof_scores, Pfa_asv=0.22276, Pmiss_asv=0.06677, Pmiss_spoof_asv=0.76843):
    """
    Menghitung minimum Tandem Detection Cost Function (min t-DCF) standar ASVspoof 2019.
    Menerapkan nilai prioritas biaya default sesuai parameter resmi.
    """
    # Parameter Biaya (Cost Parameters) untuk ASVspoof2019
    Pspoof = 0.05
    Ptar = (1.0 - Pspoof) * 0.99
    Pnon = (1.0 - Pspoof) * 0.01
    
    Cmiss_cm = 1.0
    Cfa_cm = 10.0
    Cmiss_asv = 1.0
    Cfa_asv = 10.0
    
    # Menghitung DET curve dari Countermeasure (CM)
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    y_score = np.concatenate([bonafide_scores, spoof_scores])
    
    # Hitung fpr (Pfa_cm) dan fnr (Pmiss_cm)
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    Pmiss_cm = 1.0 - tpr
    Pfa_cm = fpr
    
    # Hitung koefisien pembobot C1 dan C2
    C1 = Ptar * (Cmiss_cm - Cmiss_asv * Pmiss_asv) - Pnon * Cfa_asv * Pfa_asv
    C2 = Cfa_cm * Pspoof * (1.0 - Pmiss_spoof_asv)
    
    # Tandem cost
    tDCF = C1 * Pmiss_cm + C2 * Pfa_cm
    
    # Normalisasi t-DCF
    tDCF_norm = tDCF / np.minimum(C1, C2)
    
    # Nilai minimum t-DCF
    min_tdcf = np.min(tDCF_norm)
    return min_tdcf

# Pengujian dengan data dummy
dummy_bona = np.random.normal(loc=1.5, scale=0.8, size=500)
dummy_spoof = np.random.normal(loc=-1.5, scale=0.8, size=500)

test_eer = compute_eer_scipy(dummy_bona, dummy_spoof)
test_tdcf = compute_asvspoof19_min_tdcf(dummy_bona, dummy_spoof)

print("--- Hasil Pengujian Fungsi Evaluasi dengan Data Dummy ---")
print(f"EER      : {test_eer * 100:.4f} %")
print(f"min t-DCF: {test_tdcf:.6f}")

## 6. Training & Validation Loop
Di bawah ini adalah implementasi *training & validation loop* penuh. 
Kita menggunakan `Adam Optimizer` dan `CrossEntropyLoss`. Pada setiap akhir epoch, loop ini mengevaluasi model pada set validasi, lalu menghitung serta mencetak EER dan min t-DCF dengan format log yang rapi.

In [ ]:
import torch.optim as optim

def train_one_epoch(model, dataloader, optimizer, criterion, device, feature_extractor):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (wavs, labels) in enumerate(dataloader):
        # wavs shape: (batch, 1, max_len)
        wavs, labels = wavs.to(device), labels.to(device)
        
        # Ekstrak LFCC secara on-the-fly
        # feature_extractor menghasilkan (batch, 1, n_lfcc, time_frames)
        with torch.no_grad():
            features = feature_extractor(wavs.squeeze(1))
            
        optimizer.zero_grad()
        outputs = model(features) # Forward pass
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * wavs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

def validate_model(model, dataloader, criterion, device, feature_extractor):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Simpan skor untuk evaluasi EER dan t-DCF
    # Kita menggunakan probabilitas kelas bona fide (kelas 1) sebagai skor CM
    bonafide_scores = []
    spoof_scores = []
    
    with torch.no_grad():
        for wavs, labels in dataloader:
            wavs, labels = wavs.to(device), labels.to(device)
            features = feature_extractor(wavs.squeeze(1))
            
            outputs = model(features)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * wavs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # Ambil probabilitas/log-likelihood dari kelas bona fide (indeks 1)
            probs = F.softmax(outputs, dim=1)
            bona_probs = probs[:, 1].cpu().numpy()
            labels_np = labels.cpu().numpy()
            
            for score, lbl in zip(bona_probs, labels_np):
                if lbl == 1:
                    bonafide_scores.append(score)
                else:
                    spoof_scores.append(score)
                    
    val_loss = running_loss / total
    val_acc = (correct / total) * 100
    
    # Hitung EER dan min t-DCF
    bonafide_scores = np.array(bonafide_scores)
    spoof_scores = np.array(spoof_scores)
    
    if len(bonafide_scores) > 0 and len(spoof_scores) > 0:
        val_eer = compute_eer_scipy(bonafide_scores, spoof_scores) * 100 # %
        val_tdcf = compute_asvspoof19_min_tdcf(bonafide_scores, spoof_scores)
    else:
        val_eer = 100.0
        val_tdcf = 1.0
        
    return val_loss, val_acc, val_eer, val_tdcf

print("Fungsi Train & Validate siap digunakan.")

## 7. Eksekusi Pelatihan
Di bawah ini adalah sel untuk memulai pelatihan. Silakan sesuaikan jumlah epoch dan hiperparameter lainnya sesuai kebutuhan eksperimen Anda.

In [ ]:
# Tentukan perangkat (GPU / CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat: {device}")

# Inisialisasi ekstraktor fitur dan model wrapper
feature_extractor = LFCCFeatureExtractor(n_lfcc=20).to(device)
model = LCNNForAudioAntiSpoofing(num_classes=2).to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)

# Inisialisasi dataset dengan transform
# Menggunakan dummy dataset jika path aslinya tidak diubah/tidak ditemukan
train_dataset_lfcc = AudioASVDataset(
    protocols_file=PATH_PROTOKOL_TRAIN,
    wav_dir=PATH_WAV_DIR_TRAIN,
    max_len=64000
)

dev_dataset_lfcc = AudioASVDataset(
    protocols_file=PATH_PROTOKOL_DEV,
    wav_dir=PATH_WAV_DIR_DEV,
    max_len=64000
)

train_loader_lfcc = DataLoader(train_dataset_lfcc, batch_size=8, shuffle=True)
dev_loader_lfcc = DataLoader(dev_dataset_lfcc, batch_size=8, shuffle=False)

# Loop Pelatihan selama N Epoch
num_epochs = 5  # Silakan ubah sesuai kebutuhan riset Anda
print(f"\nMemulai Pelatihan selama {num_epochs} Epoch...\n")
print(f"{'Epoch':^8} | {'Train Loss':^12} | {'Train Acc (%)':^15} | {'Val Loss':^12} | {'Val Acc (%)':^15} | {'Val EER (%)':^15} | {'min t-DCF':^12}")
print("-" * 105)

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(
        model=model,
        dataloader=train_loader_lfcc,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        feature_extractor=feature_extractor
    )
    
    val_loss, val_acc, val_eer, val_tdcf = validate_model(
        model=model,
        dataloader=dev_loader_lfcc,
        criterion=criterion,
        device=device,
        feature_extractor=feature_extractor
    )
    
    print(f"{epoch:^8} | {train_loss:^12.4f} | {train_acc:^15.2f} | {val_loss:^12.4f} | {val_acc:^15.2f} | {val_eer:^15.4f} | {val_tdcf:^12.5f}")

print("\nPelatihan selesai!")